# 스케일, 축 및 범례 (Scales, Axes, and Legends)

시각적 인코딩 &ndash; 데이터를 위치, 크기, 모양 또는 색상과 같은 시각적 변수에 매핑하는 것 &ndash; 은 데이터 시각화의 핵심입니다. 실제로 이 매핑을 수행하는 핵심 요소는 *스케일(scale)*입니다. 스케일은 데이터 값을 입력(스케일 *도메인*)으로 받아 픽셀 위치나 RGB 색상과 같은 시각적 값을 출력(스케일 *범위*)으로 반환하는 함수입니다. 물론 아무도 그것이 무엇을 전달하는지 파악할 수 없다면 시각화는 쓸모가 없습니다! 그래픽 마크 외에도 차트에는 독자가 그래픽을 해독할 수 있게 해주는 참조 요소 또는 *가이드(guides)*가 필요합니다. *축(axes)*(공간 범위가 있는 스케일을 시각화) 및 *범례(legends)*(색상, 크기 또는 모양 범위가 있는 스케일을 시각화)와 같은 가이드는 효과적인 데이터 시각화의 숨은 영웅입니다!

이 노트북에서는 항생제의 효과에 대한 예제를 사용하여 스케일 매핑, 축 및 범례의 맞춤형 디자인을 지원하기 위해 Altair가 제공하는 옵션을 살펴보겠습니다.

_이 노트북은 [데이터 시각화 커리큘럼](https://github.com/uwdata/visualization-curriculum)의 일부입니다._

In [1]:
import pandas as pd
import altair as alt

## 항생제 데이터 (Antibiotics Data)

제2차 세계 대전 이후 항생제는 다루기 힘들었던 병에 대한 쉬운 치료제였기 때문에 "기적의 약"으로 간주되었습니다. 어떤 박테리아 감염에 어떤 약이 가장 효과적으로 작용하는지 알아보기 위해 16가지 박테리아에 대한 가장 인기 있는 3가지 항생제의 성능 데이터가 수집되었습니다.

우리는 [vega-datasets 컬렉션](https://github.com/vega/vega-datasets)의 항생제 데이터 세트를 사용할 것입니다. 아래 예제에서는 URL을 Altair에 직접 전달할 것입니다:

In [2]:
antibiotics = 'https://cdn.jsdelivr.net/npm/vega-datasets@1/data/burtin.json'

먼저 Pandas로 데이터를 로드하여 데이터 세트 전체를 보고 사용 가능한 필드를 익힐 수 있습니다:

In [3]:
pd.read_json(antibiotics)

,Bacteria,Penicillin,Streptomycin,Neomycin,Gram_Staining,Genus
0,Aerobacter aerogenes,870.000,1.00,1.600,negative,other
1,Bacillus anthracis,0.001,0.01,0.007,positive,other
2,Brucella abortus,1.000,2.00,0.020,negative,other
3,Diplococcus pneumoniae,0.005,11.00,10.000,positive,other
4,Escherichia coli,100.000,0.40,0.100,negative,other
5,Klebsiella pneumoniae,850.000,1.20,1.000,negative,other
6,Mycobacterium tuberculosis,800.000,5.00,2.000,negative,other
7,Proteus vulgaris,3.000,0.10,0.100,negative,other
8,Pseudomonas aeruginosa,850.000,2.00,0.400,negative,other
9,Salmonella (Eberthella) typhosa,1.000,0.40,0.008,negative,Salmonella


표의 수치는 항생제의 효과를 측정하는 [최소 억제 농도(MIC)](https://en.wikipedia.org/wiki/Minimum_inhibitory_concentration)를 나타내며, 이는 체외(in vitro)에서 성장을 막는 데 필요한 항생제 농도(밀리리터당 마이크로그램)를 의미합니다. [그람 염색법](https://ko.wikipedia.org/wiki/%EA%B7%B8%EB%9E%8C_%EC%97%BC%EC%83%89)이라는 절차에 대한 박테리아의 반응은 명목형 필드인 `Gram_Staining`으로 설명됩니다. 진한 파란색이나 보라색으로 변하는 박테리아는 그람 양성입니다. 그렇지 않으면 그람 음성입니다.

이 데이터 세트의 다양한 시각화를 살펴보면서 스스로 질문해 보세요: 항생제의 상대적 효과에 대해 무엇을 배울 수 있을까요? 항생제 반응을 바탕으로 박테리아 종에 대해 무엇을 배울 수 있을까요?

## 스케일 및 축 구성하기 (Configuring Scales and Axes)

### 항생제 저항성 플로팅: 스케일 유형 조정하기 (Plotting Antibiotic Resistance: Adjusting the Scale Type)

네오마이신(Neomycin)의 MIC에 대한 간단한 도트 플롯을 살펴보는 것으로 시작하겠습니다.

In [4]:
alt.Chart(antibiotics).mark_circle().encode(
    alt.X('Neomycin:Q')
)

alt.Chart(...)

*MIC 값이 여러 자릿수에 걸쳐 있음을 볼 수 있습니다. 대부분의 점은 왼쪽에 모여 있고, 오른쪽에는 몇 개의 큰 이상치가 있습니다.*

기본적으로 Altair는 도메인 값(MIC)과 범위 값(픽셀) 사이에 `linear`(선형) 매핑을 사용합니다. 데이터를 더 잘 파악하기 위해 다른 스케일 변환을 적용할 수 있습니다.

스케일 유형을 변경하려면 `alt.Scale` 메서드와 `type` 매개변수를 사용하여 `scale` 속성을 설정합니다.

다음은 제곱근(`sqrt`) 스케일 유형을 사용한 결과입니다. 이제 픽셀 범위의 거리는 데이터 도메인 거리의 제곱근에 대응합니다.

In [5]:
alt.Chart(antibiotics).mark_circle().encode(
    alt.X('Neomycin:Q',
          scale=alt.Scale(type='sqrt'))
)

alt.Chart(...)

*왼쪽의 점들이 이제 더 잘 구분되지만, 여전히 심한 왜곡이 보입니다.*

대신 [로그 스케일](https://ko.wikipedia.org/wiki/%EB%A1%9C%EA%B7%B8_%EC%B2%99%EB%8F%84)(`log`)을 사용해 보겠습니다:

In [6]:
alt.Chart(antibiotics).mark_circle().encode(
    alt.X('Neomycin:Q',
          scale=alt.Scale(type='log'))
)

alt.Chart(...)

*이제 데이터가 훨씬 더 고르게 분포되어 있으며, 서로 다른 박테리아에 필요한 농도의 매우 큰 차이를 볼 수 있습니다.*

표준 선형 스케일에서 10단위의 시각적(픽셀) 거리는 데이터 도메인에서의 10단위 *더하기*에 해당할 수 있습니다. 로그 변환은 곱셈과 덧셈 사이를 매핑하므로 `log(u) + log(v) = log(u*v)`가 성립합니다. 결과적으로 로그 스케일에서 10단위의 시각적 거리는 밑이 10인 로그를 가정할 때 데이터 도메인에서의 10단위 *곱하기*에 해당합니다. 위의 `log` 스케일은 기본적으로 밑이 10인 로그를 사용하지만, 스케일에 `base` 매개변수를 제공하여 이를 조정할 수 있습니다.

### 축 스타일 지정하기 (Styling an Axis)

복용량이 낮을수록 효과가 높음을 나타냅니다. 그러나 어떤 사람들은 "더 나은" 값이 차트 내에서 "위쪽 및 오른쪽"에 있기를 기대할 수 있습니다. 이러한 관례를 따르고 싶다면 축을 반전시켜 반전된 MIC 스케일로 "효과"를 인코딩할 수 있습니다.

이렇게 하려면 인코딩의 `sort` 속성을 `'descending'`(내림차순)으로 설정하면 됩니다:

In [7]:
alt.Chart(antibiotics).mark_circle().encode(
    alt.X('Neomycin:Q',
          sort='descending',
          scale=alt.Scale(type='log'))
)

alt.Chart(...)

*안타깝게도 축이 다소 혼란스러워지기 시작했습니다. 데이터를 로그 스케일로, 역방향으로 플로팅하고 있으며 단위가 무엇인지 명확하게 표시되지 않았습니다!*

더 유용한 정보를 담은 축 제목을 추가해 보겠습니다. 인코딩의 `title` 속성을 사용하여 원하는 제목 텍스트를 제공합니다:

In [8]:
alt.Chart(antibiotics).mark_circle().encode(
    alt.X('Neomycin:Q',
          sort='descending',
          scale=alt.Scale(type='log'),
          title='Neomycin MIC (μg/ml, reverse log scale)')
)

alt.Chart(...)

훨씬 낫네요!

기본적으로 Altair는 x축을 차트 하단에 배치합니다. 이러한 기본값을 변경하려면 `orient='top'`과 함께 `axis` 속성을 추가하면 됩니다:

In [9]:
alt.Chart(antibiotics).mark_circle().encode(
    alt.X('Neomycin:Q',
          sort='descending',
          scale=alt.Scale(type='log'),
          axis=alt.Axis(orient='top'),
          title='Neomycin MIC (μg/ml, reverse log scale)')
)

alt.Chart(...)

마찬가지로 y축은 기본적으로 `'left'`(왼쪽) 방향이지만 `'right'`(오른쪽)로 설정할 수 있습니다.

### 항생제 비교: 그리드 선, 틱 수 및 크기 조정하기 (Comparing Antibiotics: Adjusting Grid Lines, Tick Counts, and Sizing)

*네오마이신은 스트렙토마이신(streptomycin)이나 페니실린(penicillin)과 같은 다른 항생제와 어떻게 다를까요?*

이 질문에 답하기 위해 네오마이신에 대한 x축 디자인을 미러링한 다른 항생제에 대한 y축 인코딩을 추가하여 산점도를 만들 수 있습니다.

In [10]:
alt.Chart(antibiotics).mark_circle().encode(
    alt.X('Neomycin:Q',
          sort='descending',
          scale=alt.Scale(type='log'),
          title='Neomycin MIC (μg/ml, reverse log scale)'),
    alt.Y('Streptomycin:Q',
          sort='descending',
          scale=alt.Scale(type='log'),
          title='Streptomycin MIC (μg/ml, reverse log scale)')
)

alt.Chart(...)

*박테리아 균주가 두 항생제 모두에 유사하게 반응하기 때문에 네오마이신과 스트렙토마이신은 높은 상관관계가 있는 것으로 보입니다.*

계속해서 네오마이신과 페니실린을 비교해 보겠습니다:

In [11]:
alt.Chart(antibiotics).mark_circle().encode(
    alt.X('Neomycin:Q',
          sort='descending',
          scale=alt.Scale(type='log'),
          title='Neomycin MIC (μg/ml, reverse log scale)'),
    alt.Y('Penicillin:Q',
          sort='descending',
          scale=alt.Scale(type='log'),
          title='Penicillin MIC (μg/ml, reverse log scale)')
)

alt.Chart(...)

*이제 더 차별화된 반응을 볼 수 있습니다. 어떤 박테리아는 네오마이신에는 잘 반응하지만 페니실린에는 그렇지 않으며, 그 반대의 경우도 마찬가지입니다!*

이 플롯은 유용하지만 더 좋게 만들 수 있습니다. x축과 y축은 동일한 단위를 사용하지만 범위(차트 너비가 높이보다 큼)와 도메인(x축은 0.001에서 100, y축은 0.001에서 1,000)이 다릅니다.

축을 동일하게 맞춰보겠습니다. 차트에 대해 명시적인 `width` 및 `height` 설정을 추가하고, 스케일 `domain` 속성을 사용하여 일치하는 도메인을 지정할 수 있습니다.

In [12]:
alt.Chart(antibiotics).mark_circle().encode(
    alt.X('Neomycin:Q',
          sort='descending',
          scale=alt.Scale(type='log', domain=[0.001, 1000]),
          title='Neomycin MIC (μg/ml, reverse log scale)'),
    alt.Y('Penicillin:Q',
          sort='descending',
          scale=alt.Scale(type='log', domain=[0.001, 1000]),
          title='Penicillin MIC (μg/ml, reverse log scale)')
).properties(width=250, height=250)

alt.Chart(...)

*결과 플롯은 더 균형 잡혀 있으며, 미묘한 오해를 불러일으킬 가능성이 적습니다!*

하지만 이제 그리드 선이 다소 촘촘합니다. 그리드 선을 완전히 제거하려면 `axis` 속성에 `grid=False`를 추가하면 됩니다. 하지만 대신 틱 마크의 수를 줄여서 예를 들어 각 자릿수(order of magnitude)에 대해서만 그리드 선을 포함하고 싶다면 어떻게 해야 할까요?

틱의 수를 변경하려면 `Axis` 객체에 대해 대상 `tickCount` 속성을 지정할 수 있습니다. `tickCount`는 Altair에게 하나의 *제안*으로 취급되며, 사람이 읽기 좋은 간격을 사용하는 것과 같은 다른 측면과 함께 고려됩니다. 요청한 틱 마크 수와 *정확히* 일치하지 않을 수도 있지만 비슷한 결과를 얻을 수 있을 것입니다.

In [13]:
alt.Chart(antibiotics).mark_circle().encode(
    alt.X('Neomycin:Q',
          sort='descending',
          scale=alt.Scale(type='log', domain=[0.001, 1000]),
          axis=alt.Axis(tickCount=5),
          title='Neomycin MIC (μg/ml, reverse log scale)'),
    alt.Y('Penicillin:Q',
          sort='descending',
          scale=alt.Scale(type='log', domain=[0.001, 1000]),
          axis=alt.Axis(tickCount=5),
          title='Penicillin MIC (μg/ml, reverse log scale)')
).properties(width=250, height=250)

alt.Chart(...)

`tickCount`를 5로 설정하여 원하는 효과를 얻었습니다.

산점도의 점들이 조금 작게 느껴집니다. 원 마크의 `size` 속성을 설정하여 기본 크기를 변경해 보겠습니다. 이 크기 값은 픽셀 단위의 마크 *면적*입니다.

In [14]:
alt.Chart(antibiotics).mark_circle(size=80).encode(
    alt.X('Neomycin:Q',
          sort='descending',
          scale=alt.Scale(type='log', domain=[0.001, 1000]),
          axis=alt.Axis(tickCount=5),
          title='Neomycin MIC (μg/ml, reverse log scale)'),
    alt.Y('Penicillin:Q',
          sort='descending',
          scale=alt.Scale(type='log', domain=[0.001, 1000]),
          axis=alt.Axis(tickCount=5),
          title='Penicillin MIC (μg/ml, reverse log scale)'), 
).properties(width=250, height=250)

alt.Chart(...)

여기서는 원 마크 면적을 80 픽셀로 설정했습니다. *원하는 대로 값을 더 조정해 보세요!*

## 색상 범례 구성하기 (Configuring Color Legends)

### 그람 염색법에 따른 색상 (Color by Gram Staining)

*위에서 우리는 네오마이신이 어떤 박테리아에는 더 효과적이고, 페니실린은 다른 박테리아에 더 효과적이라는 것을 보았습니다. 하지만 박테리아의 구체적인 종을 모른다면 어떤 항생제를 사용해야 할지 어떻게 알 수 있을까요? 그람 염색법은 박테리아의 종류를 구별하기 위한 진단법 역할을 합니다!*

`Gram_Staining`을 명목형 데이터 유형으로 `color` 채널에 인코딩해 보겠습니다:

In [15]:
alt.Chart(antibiotics).mark_circle(size=80).encode(
    alt.X('Neomycin:Q',
          sort='descending',
          scale=alt.Scale(type='log', domain=[0.001, 1000]),
          axis=alt.Axis(tickCount=5),
          title='Neomycin MIC (μg/ml, reverse log scale)'),
    alt.Y('Penicillin:Q',
          sort='descending',
          scale=alt.Scale(type='log', domain=[0.001, 1000]),
          axis=alt.Axis(tickCount=5),
          title='Penicillin MIC (μg/ml, reverse log scale)'),
    alt.Color('Gram_Staining:N')
).properties(width=250, height=250)

alt.Chart(...)

*그람 양성(Gram-positive) 박테리아는 페니실린에 가장 민감한 것으로 보이며, 네오마이신은 그람 음성(Gram-negative) 박테리아에 더 효과적인 것 같습니다!*

위의 색상 체계는 명목형(같음 또는 같지 않음) 비교를 위해 지각적으로 구별 가능한 색상을 제공하도록 자동으로 선택되었습니다. 그러나 사용되는 색상을 사용자 정의하고 싶을 수도 있습니다. 이 경우 그람 염색법은 [독특한 물리적 색상 결과(그람 음성은 분홍색, 그람 양성은 보라색)](https://ko.wikipedia.org/wiki/%EA%B7%B8%EB%9E%8C_%EC%97%BC%EC%83%89#/media/%ED%8C%8C%EC%9D%BC:Gram_stain_01.jpg)를 나타냅니다.

데이터 `domain`(도메인)에서 색상 `range`(범위)로 명시적인 스케일 매핑을 지정하여 해당 색상을 사용해 보겠습니다:

In [16]:
alt.Chart(antibiotics).mark_circle(size=80).encode(
    alt.X('Neomycin:Q',
          sort='descending',
          scale=alt.Scale(type='log', domain=[0.001, 1000]),
          axis=alt.Axis(tickCount=5),
          title='Neomycin MIC (μg/ml, reverse log scale)'),
    alt.Y('Penicillin:Q',
          sort='descending',
          scale=alt.Scale(type='log', domain=[0.001, 1000]),
          axis=alt.Axis(tickCount=5),
          title='Penicillin MIC (μg/ml, reverse log scale)'),
    alt.Color('Gram_Staining:N',
          scale=alt.Scale(domain=['negative', 'positive'], range=['hotpink', 'purple'])
    )
).properties(width=250, height=250)

alt.Chart(...)

기본적으로 범례는 차트 오른쪽에 배치됩니다. 축과 마찬가지로 `orient` 매개변수를 사용하여 범례의 방향을 변경할 수 있습니다:

In [17]:
alt.Chart(antibiotics).mark_circle(size=80).encode(
    alt.X('Neomycin:Q',
          sort='descending',
          scale=alt.Scale(type='log', domain=[0.001, 1000]),
          axis=alt.Axis(tickCount=5),
          title='Neomycin MIC (μg/ml, reverse log scale)'),
    alt.Y('Penicillin:Q',
          sort='descending',
          scale=alt.Scale(type='log', domain=[0.001, 1000]),
          axis=alt.Axis(tickCount=5),
          title='Penicillin MIC (μg/ml, reverse log scale)'),
    alt.Color('Gram_Staining:N',
          scale=alt.Scale(domain=['negative', 'positive'], range=['hotpink', 'purple']),
          legend=alt.Legend(orient='left')
    )
).properties(width=250, height=250)

alt.Chart(...)

`legend=None`을 지정하여 범례를 완전히 제거할 수도 있습니다:

In [18]:
alt.Chart(antibiotics).mark_circle(size=80).encode(
    alt.X('Neomycin:Q',
          sort='descending',
          scale=alt.Scale(type='log', domain=[0.001, 1000]),
          axis=alt.Axis(tickCount=5),
          title='Neomycin MIC (μg/ml, reverse log scale)'),
    alt.Y('Penicillin:Q',
          sort='descending',
          scale=alt.Scale(type='log', domain=[0.001, 1000]),
          axis=alt.Axis(tickCount=5),
          title='Penicillin MIC (μg/ml, reverse log scale)'),
    alt.Color('Gram_Staining:N',
          scale=alt.Scale(domain=['negative', 'positive'], range=['hotpink', 'purple']),
          legend=None
    )
).properties(width=250, height=250)

alt.Chart(...)

### 종(Species)에 따른 색상 (Color by Species)

*지금까지 우리는 항생제의 효과를 고려했습니다. 이제 방향을 바꿔서 다른 질문을 해보겠습니다: 항생제 반응이 박테리아의 다른 종에 대해 무엇을 가르쳐줄 수 있을까요?*

우선, `color` 채널을 사용하여 `Bacteria`(명목형 데이터 필드)를 인코딩해 보겠습니다:

In [19]:
alt.Chart(antibiotics).mark_circle(size=80).encode(
    alt.X('Neomycin:Q',
          sort='descending',
          scale=alt.Scale(type='log', domain=[0.001, 1000]),
          axis=alt.Axis(tickCount=5),
          title='Neomycin MIC (μg/ml, reverse log scale)'),
    alt.Y('Penicillin:Q',
          sort='descending',
          scale=alt.Scale(type='log', domain=[0.001, 1000]),
          axis=alt.Axis(tickCount=5),
          title='Penicillin MIC (μg/ml, reverse log scale)'),
    alt.Color('Bacteria:N')
).properties(width=250, height=250)

alt.Chart(...)

*결과가 다소 엉망입니다!* 박테리아의 종류가 너무 많아서 Altair가 명목형 값을 위한 기본 10색 팔레트에서 색상을 반복하기 시작합니다.

사용자 정의 색상을 사용하기 위해 색상 인코딩 `scale` 속성을 업데이트할 수 있습니다. 한 가지 옵션은 그람 염색법에서 했던 것처럼 값당 정확한 색상 매핑을 나타내기 위해 명시적인 스케일 `domain` 및 `range` 값을 제공하는 것입니다. 또 다른 옵션은 대체 색상 체계를 사용하는 것입니다. Altair에는 다양한 내장 색상 체계가 포함되어 있습니다. 전체 목록은 [Vega 색상 체계 문서](https://vega.github.io/vega/docs/schemes/#reference)를 참조하세요.

내장된 20색 체계인 `tableau20`으로 전환해 보고, 스케일 `scheme` 속성을 사용하여 설정해 보겠습니다.

In [20]:
alt.Chart(antibiotics).mark_circle(size=80).encode(
    alt.X('Neomycin:Q',
          sort='descending',
          scale=alt.Scale(type='log', domain=[0.001, 1000]),
          axis=alt.Axis(tickCount=5),
          title='Neomycin MIC (μg/ml, reverse log scale)'),
    alt.Y('Penicillin:Q',
          sort='descending',
          scale=alt.Scale(type='log', domain=[0.001, 1000]),
          axis=alt.Axis(tickCount=5),
          title='Penicillin MIC (μg/ml, reverse log scale)'),
    alt.Color('Bacteria:N',
          scale=alt.Scale(scheme='tableau20'))
).properties(width=250, height=250)

alt.Chart(...)

*이제 각 박테리아에 대해 고유한 색상을 갖게 되었지만 차트는 여전히 복잡합니다. 다른 문제들 중에서, 이 인코딩은 같은 속(genus)에 속하는 박테리아를 고려하지 않습니다. 위의 차트에서 두 개의 다른 살모넬라(Salmonella) 균주는 생물학적 사촌임에도 불구하고 매우 다른 색조(청록색과 분홍색)를 가지고 있습니다.*

다른 체계를 시도해 보기 위해 데이터 유형을 명목형(nominal)에서 서수형(ordinal)으로 변경할 수도 있습니다. 기본 서수형 체계는 밝은 파란색에서 어두운 파란색으로 변하는 파란색 음영을 사용합니다:

In [21]:
alt.Chart(antibiotics).mark_circle(size=80).encode(
    alt.X('Neomycin:Q',
          sort='descending',
          scale=alt.Scale(type='log', domain=[0.001, 1000]),
          axis=alt.Axis(tickCount=5),
          title='Neomycin MIC (μg/ml, reverse log scale)'),
    alt.Y('Penicillin:Q',
          sort='descending',
          scale=alt.Scale(type='log', domain=[0.001, 1000]),
          axis=alt.Axis(tickCount=5),
          title='Penicillin MIC (μg/ml, reverse log scale)'),
    alt.Color('Bacteria:O')
).properties(width=250, height=250)

alt.Chart(...)

*파란색 음영 중 일부는 구별하기 어려울 수 있습니다.*

더 차별화된 색상을 위해 기본 `blues` 색상 체계의 대안을 실험해 볼 수 있습니다. `viridis` 체계는 색조(hue)와 휘도(luminance)를 모두 변화시킵니다:

In [22]:
alt.Chart(antibiotics).mark_circle(size=80).encode(
    alt.X('Neomycin:Q',
          sort='descending',
          scale=alt.Scale(type='log', domain=[0.001, 1000]),
          axis=alt.Axis(tickCount=5),
          title='Neomycin MIC (μg/ml, reverse log scale)'),
    alt.Y('Penicillin:Q',
          sort='descending',
          scale=alt.Scale(type='log', domain=[0.001, 1000]),
          axis=alt.Axis(tickCount=5),
          title='Penicillin MIC (μg/ml, reverse log scale)'),
    alt.Color('Bacteria:O',
          scale=alt.Scale(scheme='viridis'))
).properties(width=250, height=250)

alt.Chart(...)

*같은 속(genus)에 속하는 박테리아들이 이전보다 더 유사한 색상을 가지게 되었지만, 차트는 여전히 혼란스럽습니다. 색상이 너무 많고, 범례에서 정확하게 찾아보기가 어려우며, 두 박테리아가 비슷한 색상을 가졌더라도 속이 다를 수 있습니다.*

### 속(Genus)에 따른 색상 (Color by Genus)

박테리아 개별 종 대신 속(genus)별로 색상을 지정해 보겠습니다. 이를 위해 박테리아 이름을 공백 문자로 분할하고 결과 배열에서 첫 번째 단어를 가져오는 `calculate` 변환을 추가하겠습니다. 그런 다음 결과물인 `Genus` 필드를 `tableau20` 색상 체계를 사용하여 인코딩할 수 있습니다.

(항생제 데이터 세트에는 미리 계산된 `Genus` 필드가 포함되어 있지만, Altair의 데이터 변환을 더 자세히 살펴보기 위해 여기서는 무시하겠습니다.)

In [23]:
alt.Chart(antibiotics).mark_circle(size=80).transform_calculate(
    Genus='split(datum.Bacteria, " ")[0]'
).encode(
    alt.X('Neomycin:Q',
          sort='descending',
          scale=alt.Scale(type='log', domain=[0.001, 1000]),
          axis=alt.Axis(tickCount=5),
          title='Neomycin MIC (μg/ml, reverse log scale)'),
    alt.Y('Penicillin:Q',
          sort='descending',
          scale=alt.Scale(type='log', domain=[0.001, 1000]),
          axis=alt.Axis(tickCount=5),
          title='Penicillin MIC (μg/ml, reverse log scale)'),
    alt.Color('Genus:N',
          scale=alt.Scale(scheme='tableau20'))
).properties(width=250, height=250)

alt.Chart(...)

*흠... 데이터가 속에 의해 더 잘 분리되기는 했지만, 이 수많은 색상의 향연은 특별히 유용해 보이지는 않습니다.*

*이전 차트 중 일부를 주의 깊게 살펴보면 살모넬라(Salmonella), 포도상구균(Staphylococcus), 연쇄상구균(Streptococcus)과 같이 소수의 박테리아만이 다른 박테리아와 속을 공유한다는 것을 알 수 있습니다. 비교를 집중시키기 위해 이러한 반복되는 속 값에 대해서만 색상을 추가할 수 있습니다.*

속 이름을 가져와서 반복되는 값 중 하나이면 그대로 유지하고, 그렇지 않으면 `"Other"`(기타)라는 문자열을 사용하는 또 다른 `calculate` 변환을 추가해 보겠습니다.

또한 색상 인코딩 `scale`에 대해 명시적인 `domain` 및 `range` 배열을 사용하여 사용자 정의 색상 인코딩을 추가할 수 있습니다.


In [24]:
alt.Chart(antibiotics).mark_circle(size=80).transform_calculate(
  Split='split(datum.Bacteria, " ")[0]'
).transform_calculate(
  Genus='indexof(["Salmonella", "Staphylococcus", "Streptococcus"], datum.Split) >= 0 ? datum.Split : "Other"'
).encode(
    alt.X('Neomycin:Q',
          sort='descending',
          scale=alt.Scale(type='log', domain=[0.001, 1000]),
          axis=alt.Axis(tickCount=5),
          title='Neomycin MIC (μg/ml, reverse log scale)'),
    alt.Y('Penicillin:Q',
          sort='descending',
          scale=alt.Scale(type='log', domain=[0.001, 1000]),
          axis=alt.Axis(tickCount=5),
          title='Penicillin MIC (μg/ml, reverse log scale)'),
    alt.Color('Genus:N',
          scale=alt.Scale(
            domain=['Salmonella', 'Staphylococcus', 'Streptococcus', 'Other'],
            range=['rgb(76,120,168)', 'rgb(84,162,75)', 'rgb(228,87,86)', 'rgb(121,112,110)']
          ))
).properties(width=250, height=250)

alt.Chart(...)

*이제 축과 범례를 사용자 정의함으로써 훨씬 더 많은 것을 드러내는 플롯을 갖게 되었습니다. 잠시 시간을 내어 위의 플롯을 살펴보세요. 놀라운 그룹화가 보이나요?*

*왼쪽 상단 지역에는 빨간색 Streptococcus 박테리아 무리가 있지만, 회색 Other 박테리아가 그들과 함께 있습니다. 한편, 오른쪽 중간쯤에는 "사촌"들과 멀리 떨어진 또 다른 빨간색 Streptococcus가 보입니다. 같은 속의 박테리아(따라서 아마도 유전적으로 더 유사할 것임)가 더 가깝게 그룹화될 것이라고 기대할 수 있지 않을까요?*

알고 보니 기초가 되는 데이터 세트에 실제로 오류가 포함되어 있습니다. 이 데이터 세트는 1950년대 초에 사용된 종 지정을 반영합니다. 그러나 그 이후로 과학적 합의가 뒤집혔습니다. 왼쪽 상단의 그 회색 점은? 이제 Streptococcus로 간주됩니다! 오른쪽 중간의 그 빨간 점은? 더 이상 Streptococcus로 간주되지 않습니다!

물론 이 데이터 세트 자체만으로는 이러한 재분류를 완전히 정당화할 수 없습니다. 그럼에도 불구하고, 데이터에는 수십 년 동안 간과되었던 귀중한 생물학적 단서가 포함되어 있습니다! 시각화는 적절한 기술과 탐구심을 가진 관찰자가 사용할 때 발견을 위한 강력한 도구가 될 수 있습니다.

이 예제는 또한 중요한 교훈을 강조합니다: **항상 당신의 데이터를 의심하십시오! (always be skeptical of your data!)**

### 항생제 반응에 따른 색상 (Color by Antibiotic Response)

`color` 채널을 사용하여 정량적 값을 인코딩할 수도 있습니다. 하지만 일반적으로 색상은 위치나 크기 인코딩만큼 수량을 전달하는 데 효과적이지 않다는 점을 명심하세요!

다음은 각 박테리아에 대한 페니실린 MIC 값의 기본적인 히트맵입니다. `rect` 마크를 사용하고 박테리아를 MIC 값의 내림차순(저항성이 가장 높은 것부터 낮은 것 순서)으로 정렬해 보겠습니다:

In [25]:
alt.Chart(antibiotics).mark_rect().encode(
    alt.Y('Bacteria:N',
      sort=alt.EncodingSortField(field='Penicillin', op='max', order='descending')
    ),
    alt.Color('Penicillin:Q')
)

alt.Chart(...)

로그 변환된 스케일, 축 방향 변경, 사용자 정의 색상 체계(`plasma`), 틱 수 조정 및 사용자 정의 제목 텍스트 등 지금까지 살펴본 기능을 결합하여 이 차트를 더욱 개선할 수 있습니다. 또한 축 제목 배치와 범례 제목 정렬을 조정하기 위해 구성(configuration) 옵션을 사용할 것입니다.

In [26]:
alt.Chart(antibiotics).mark_rect().encode(
    alt.Y('Bacteria:N',
      sort=alt.EncodingSortField(field='Penicillin', op='max', order='descending'),
      axis=alt.Axis(
        orient='right',     # 차트의 오른쪽에 축 배치
        titleX=7,           # 차트의 오른쪽 7픽셀로 x 위치 설정
        titleY=-2,          # 차트의 위쪽 2픽셀로 y 위치 설정
        titleAlign='left',  # 왼쪽 정렬된 텍스트 사용
        titleAngle=0        # 기본 제목 회전 취소
      )
    ),
    alt.Color('Penicillin:Q',
      scale=alt.Scale(type='log', scheme='plasma', nice=True),
      legend=alt.Legend(titleOrient='right', tickCount=5),
      title='Penicillin MIC (μg/ml)'
    )
)

alt.Chart(...)

또는 축 제목을 완전히 제거하고 최상위 `title` 속성을 사용하여 전체 차트에 대한 제목을 추가할 수 있습니다:

In [27]:
alt.Chart(antibiotics, title='박테리아 균주의 페니실린 저항성').mark_rect().encode(
    alt.Y('Bacteria:N',
      sort=alt.EncodingSortField(field='Penicillin', op='max', order='descending'),
      axis=alt.Axis(orient='right', title=None)
    ),
    alt.Color('Penicillin:Q',
      scale=alt.Scale(type='log', scheme='plasma', nice=True),
      legend=alt.Legend(titleOrient='right', tickCount=5),
      title='Penicillin MIC (μg/ml)'
    )
).configure_title(
  anchor='start', # 제목 고정 및 왼쪽 정렬
  offset=5        # 차트에서 제목 오프셋 설정
)

alt.Chart(...)

## 요약 (Summary)

지금까지 여러 노트북을 통해 배운 인코딩, 데이터 변환 및 사용자 정의에 관한 내용을 통합하면, 이제 다양한 통계 그래픽을 만들 준비가 되었을 것입니다. 이제 데이터를 탐색하고 전달하는 일상적인 작업에 Altair를 활용할 수 있습니다!

이 주제에 대해 더 자세히 알고 싶으신가요?

- [Altair 시각화 사용자 정의 문서](https://altair-viz.github.io/user_guide/customization.html)부터 시작하세요.
- 스케일 매핑에 대한 보충 설명은 ["Introducing d3-scale"](https://medium.com/@mbostock/introducing-d3-scale-61980c51545f)을 참조하세요.
- Altair 및 Vega-Lite의 기반이 되는 Vega 라이브러리에서 축과 범례를 스타일링하는 모든 방법에 대한 심층적인 탐구는 ["A Guide to Guides: Axes & Legends in Vega"](https://beta.observablehq.com/@jheer/a-guide-to-guides-axes-legends-in-vega)를 참조하세요.
- 항생제 데이터 세트의 매혹적인 역사는 *American Scientist*의 [Wainer & Lysen의 "That's Funny..."](https://www.americanscientist.org/article/thats-funny)를 참조하세요.